In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('combined_harpy.csv')

In [3]:
# Sort by time (important for sliding window)
df = df.sort_values('time').reset_index(drop=True)

In [4]:
# Create sliding window features (window = 10 samples ≈ 100ms)
window = 10

In [5]:
def create_features(group):
    group = group.copy()
    for col in ['branch-misses', 'branch-instructions', 'LLC-load-misses', 
                'L1-dcache-load-misses', 'instructions']:
        group[f'{col}_mean'] = group[col].rolling(window).mean()
        group[f'{col}_std'] = group[col].rolling(window).std()
        group[f'{col}_max'] = group[col].rolling(window).max()
    
    # Important ratios (these catch attacks really well)
    group['miss_rate'] = group['LLC-load-misses'] / (group['instructions'] + 1)
    group['branch_miss_rate'] = group['branch-misses'] / (group['branch-instructions'] + 1)
    group['cache_miss_ratio'] = group['L1-dcache-load-misses'] / (group['LLC-load-misses'] + 1)
    
    return group

In [6]:
# Apply to whole dataset
print("Creating features... (this takes ~30 seconds)")
engineered = create_features(df)

Creating features... (this takes ~30 seconds)


In [7]:
# Drop rows with NaN (first few rows of each window)
engineered = engineered.dropna().reset_index(drop=True)

In [8]:
# Save
engineered.to_csv('engineered_features.csv', index=False)
print("✅ Feature engineering done! Saved as engineered_features.csv")
print(f"Shape: {engineered.shape}")

✅ Feature engineering done! Saved as engineered_features.csv
Shape: (39941, 26)
